# Short Time Fourier Transform

- author Sylvie Dagoret-Campagne
- creation date August 19th 2020


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import host_subplot
import mpl_toolkits.axisartist as AA
import os,sys

import matplotlib.colors as colors
import matplotlib.cm as cmx

from astropy.io import fits
import matplotlib.colors as colors
import matplotlib.cm as cmx
import matplotlib.dates as mdates
from matplotlib import gridspec
%matplotlib inline

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
sys.path.append("../../tools/atmanalyticsim") # go to parent dir

In [ ]:
import libatmscattering as atm

In [ ]:
import pywt

In [ ]:
# to enlarge the sizes
params = {'legend.fontsize': 'x-large',
          'figure.figsize': (12, 8),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
plt.rcParams.update(params)
plt.rcParams['font.size'] = 18

In [ ]:
if "setup_text_plots" not in globals():
    from astroML.plotting import setup_text_plots
setup_text_plots(fontsize=18, usetex=True)

In [ ]:
def thresh_hard_sparse(x, k):
    """
    Keep only k largest entries of x and return their indices.
    Parameters
    ----------
    x : numpy array
        Numpy array to be thresholded
    k : int
        Number of largest entries in absolute value to keep
    Notes
    """
    _x = x.copy()
    ind = np.argpartition(abs(_x), -k, axis=None)[-k:]
    ind = np.unravel_index(ind, _x.shape)
    ind_del = np.ones(_x.shape, dtype=bool)
    ind_del[ind] = False
    _x[ind_del] = 0
    return ind, _x

In [ ]:
def index_to_level(indexes,coeffs_shapes):
    """
    Compute the level number corresponding to the indices
    Parameters:
    ----------
    indexes : numpy array of indices
    coeffs_shape : structure of coefficients of pywt from 
    
    return the level number of each indexes (numpy array)
 
    """
    
    
    nlevels=len(coeffs_shapes)
    sum_nb_coeffs=np.zeros(nlevels)

    # first compute cumulated threshold indexes over level
    for ilevel in np.arange(nlevels):
        if ilevel==0:
            sum_nb_coeffs[ilevel]=coeffs_shapes[ilevel][0]
        else:
            sum_nb_coeffs[ilevel]=sum_nb_coeffs[ilevel-1]+coeffs_shapes[ilevel]['d'][0]
            
  
            
    # for each index compute the level
    indexes_levels=np.zeros_like(indexes)
    
    for ilevel in np.arange(nlevels):
       
        if ilevel==0:
            #indexes_levels=np.where(indexes<int(sum_nb_coeffs[ilevel]),ilevel,0)
            sel_indexes=np.where(indexes<int(sum_nb_coeffs[ilevel]))
            indexes_levels[sel_indexes[0]]=ilevel                     
        else:
            #indexes_levels=np.where(np.logical_and(indexes>=int(sum_nb_coeffs[ilevel-1]),
            #                                       indexes<=int(sum_nb_coeffs[ilevel]))
            #                                       ,ilevel,0)
            sel_indexes=np.where(
                np.logical_and(indexes>=int(sum_nb_coeffs[ilevel-1]),
                               indexes<int(sum_nb_coeffs[ilevel])))
            indexes_levels[sel_indexes[0]]=ilevel                          
                                                   
    return indexes_levels

In [ ]:
DATADIR="../../data/atm"

In [ ]:
atmospheric_basename_files=os.listdir(DATADIR)

In [ ]:
atmospheric_basename_files

In [ ]:
input_file=os.path.join(DATADIR,'lsst_atm_10year_bigimg.fits')

In [ ]:
hdu = fits.open(input_file)

In [ ]:
hdr=hdu[0].header
data=hdu[0].data

In [ ]:
hdr

In [ ]:
NbAtmSimul=hdr['NBATMSIM']
idx_out_num=hdr['ID_NUM']
idx_out_year=hdr['ID_YEAR']
idx_out_am=hdr['ID_AM']
idx_out_vaod=hdr['ID_VAOD']
idx_out_pwv=hdr['ID_PWV']
idx_out_o3=hdr['ID_O3']
idx_out_cld=hdr['ID_CLD']
idx_out_res=hdr['ID_RES']

In [ ]:
num=data[1:,idx_out_num]
year=data[1:,idx_out_year]
airmass=data[1:,idx_out_year]
vaod=data[1:,idx_out_vaod] # vertical aerosol depth
pwv=data[1:,idx_out_pwv]   # precipitable water vapor (mm)
o3=data[1:,idx_out_o3]     # ozone
cld=data[1:,idx_out_cld]   # clouds (not used)

In [ ]:
# Extract wavelength Wavelength
wl=data[0,idx_out_res:]
transm=data[1:,idx_out_res:]

In [ ]:
NWL=wl.shape[0]

In [ ]:
jet = plt.get_cmap('jet')
cNorm = colors.Normalize(vmin=0, vmax=NWL)
scalarMap = cmx.ScalarMappable(norm=cNorm, cmap=jet)
all_colors = scalarMap.to_rgba(np.arange(NWL), alpha=1)

# Prepare data

In [ ]:
vaodarr=vaod[:,np.newaxis]
pwvarr=pwv[:,np.newaxis]
o3arr=o3[:,np.newaxis]

In [ ]:
Y=np.concatenate((vaodarr,pwvarr,o3arr),axis=1)

In [ ]:
WLMINSEL=340.
WLMAXSEL=1100.

In [ ]:
indexes_selected=np.where(np.logical_and(wl>=WLMINSEL,wl<=WLMAXSEL))[0]

In [ ]:
# need even number of bins
if len(indexes_selected)%2:
    indexes_selected=indexes_selected[:-1]

In [ ]:
len(indexes_selected)

In [ ]:
wl=wl[indexes_selected]
transm=transm[:,indexes_selected]

In [ ]:
WLMIN=wl[0]
WLMAX=wl[-1]

In [ ]:
index_atm=2

In [ ]:
#transm0=transm[0]*0.96
transm0=transm[index_atm]
vaod=vaodarr[index_atm]

In [ ]:
od=atm.RayOptDepth_adiabatic(wl, altitude=atm.altitude0, costh=1/1.2)
att_rayleigh=np.exp(-od)

In [ ]:
ax = plt.subplot(111)
ax.plot(wl,-2.5*np.log10(transm0),'b-',label="transmission")
ax.plot(wl,-2.5*np.log10(att_rayleigh),'r-',label="rayleigh attenuation")
ax.plot(wl,-2.5*np.log10(transm0/att_rayleigh),'g:',label="transmission - no Rayleigh")
ax.legend()
ax.grid()
ax.set_xlabel("$\lambda$ (nm)")
ax.set_ylabel("att (mag)")

In [ ]:
#fonction math
import math
from scipy import signal

In [ ]:
xmin=wl[0]
xmax=wl[-1]
fs=len(wl)/(xmax-xmin)  # in nm-1
print("fs/2=:",fs/2)
NFFT=32
cut = 0
#window = np.hanning(NFFT) # default
window = signal.windows.boxcar(NFFT) # default

ts = np.arange(xmin,xmax,1./fs) # plus simple a comprendre qur linespace



xts = -2.5*np.log10(transm0/att_rayleigh)

#freqs[0]: donne la composante continue
freqs, tps, Zxx = signal.stft(xts, fs=fs, nperseg=NFFT,noverlap=0,
                              window=window, boundary=None,padded=False)
spec = np.abs(Zxx)
spmean=np.max(spec[1:,:],axis=0)

In [ ]:
fig = plt.figure(figsize=(20,10))
ax=fig.add_subplot(311)
for idx in np.arange(Zxx.shape[0]):
    ax.plot(tps,Zxx[idx,:])
ax.set_title("Zxx")
ax=fig.add_subplot(312)
for idx in np.arange(spec.shape[0]):
    ax.plot(tps,spec[idx,:])
ax.set_title("spec")
ax=fig.add_subplot(313)
for idx in np.arange(spmean.shape[0]):
    ax.plot(tps,spmean)
ax.set_title("spmean")
    

In [ ]:
fig = plt.figure(figsize=(10,10))
gs = plt.GridSpec(3, 1, hspace=0.1, height_ratios=[1,2/3,2/3])
ax0 = fig.add_subplot(gs[0,0])
ax0.plot(ts,xts)
ax0.set_ylabel('Signal [u.a]')
ax0.set_xlim([xmin,xmax])
ax0.set_xticks([])
ax0.grid()
#
#
ax1 = fig.add_subplot(gs[1,0])
plt.pcolormesh(xmin+tps, freqs, 20*np.log10(spec))
ax1.plot([xmin, xmax], [freqs[1], freqs[1]], color='k', linestyle='-', linewidth=1)
ax1.plot([xmin, xmax], [freqs[-cut], freqs[-cut]], color='k', linestyle='-', linewidth=1)
ax1.set_ylabel('Freq. [$nm^{-1}$]')
ax1.set_xlim([xmin,xmax])
ax1.set_xticks([])
#
ax2 = fig.add_subplot(gs[2,0])
ax2.plot(xmin+tps,20.*np.log10(spmean))
ax2.set_xlim([xmin,xmax])
#ax2.set_ylabel('Pwr_cut dB')
ax2.set_xlabel('$\lambda$ [nm]')
ax2.grid()
#plt.colorbar(cax,ax=ax2).set_label('Intensity [dB]')
plt.show() 